# All-sky multimodal DHI — Colab GPU training

Trains the **multimodal** all-sky pipeline (V0–V7 model ladder) that estimates
diffuse horizontal irradiance (DHI) + clear-sky index + sky class from a sky
image (DINOv2 embedding or end-to-end) plus non-radiometric sensor context.

This notebook is **thin**: it clones the repo, provisions a CPython 3.14 venv
with `uv`, unpacks a prepared **Colab bundle** (manifest + splits + embeddings),
then drives the `allsky` CLI. All business logic lives in the package — the only
things to edit are in the **CONFIG** cell below.

**Runtime → Change runtime type → GPU (a T4 is enough)** before running. The
default experiment (`v4_film`) trains on precomputed embeddings and is light.

> This notebook has **not** been executed or validated on a real Colab runtime
> from the development environment — treat exact versions/timings as best-effort.


In [ ]:
!nvidia-smi -L

In [ ]:
# ============================== CONFIG (edit me) ==============================
# Every knob for this notebook lives here (env-var style). Nothing below needs
# editing for a standard run.
REPO_URL = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = "feat/allsky-multimodal"
WORKDIR = "/content/micrometeorology"  # repo clone location

# The experiment to run — any file under configs/allsky/experiments/:
#   v0_climatology  v1_sensor_only  v2_image_only  v3_concat
#   v4_film (default)  v5_multitask  v6_film_finetune (image mode)  v7_cross_attention
EXPERIMENT_CONFIG = "configs/allsky/experiments/v4_film.yaml"

# Prepared Colab bundle (allsky export-colab-bundle output): a Drive path or an
# uploaded file. Contains manifest.parquet + .meta.json, splits.json, embeddings/
# and the configs used.
BUNDLE_PATH = "/content/drive/MyDrive/labmim/allsky-mm/bundle.tar.gz"

DATA_ROOT = "/content/allsky-mm/data"  # bundle unpacks here (fast local disk)
# export-colab-bundle nests everything under a single top-level allsky_bundle/
# dir, so the manifest, splits and embeddings actually live under BUNDLE_ROOT:
BUNDLE_ROOT = f"{DATA_ROOT}/allsky_bundle"  # --manifest / --data-root target
OUTPUT_DIR = "/content/allsky-mm/out"  # run dir: checkpoints, metrics, TB
DRIVE_DIR = "/content/drive/MyDrive/labmim/runs/allsky-mm"  # persisted backup target
DRIVE_MOUNT = True  # mount Drive for the bundle + backups
# =============================================================================

In [ ]:
# Clone-or-pull the repository at the requested branch.
import os
import subprocess

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)
print("repo ready at", WORKDIR)

In [ ]:
# Bootstrap a CPython 3.14 venv with uv and install the package into it.
#
# WHY a uv-provisioned CPython 3.14 venv instead of the base Colab Python:
# this package sets requires-python>=3.14, and the Colab runtime's Python is NOT
# assumed to be 3.14 (today it usually is not). Installing `uv`, provisioning a
# standalone CPython 3.14, and installing into a dedicated venv makes this
# notebook work on ANY Colab runtime regardless of its own Python version.
#
# NOTE: not executed/validated on a real Colab runtime from the dev environment.
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(
    ["uv", "sync", "--locked", "--extra", "allsky"],
    cwd=WORKDIR,
    check=True,
)

# The [allsky] extra pins a CPU torch wheel. For real GPU acceleration replace it
# with the matching CUDA build in the SAME venv (uncomment if needed):
# subprocess.run(
#     ["uv", "pip", "install", "--python", ".venv/bin/python",
#      "--reinstall", "--torch-backend", "cu130", "torch==2.13.0"],
#     cwd=WORKDIR, check=True,
# )

ALLSKY = f"{WORKDIR}/.venv/bin/allsky"  # CLI entry point inside the venv
print("allsky CLI:", ALLSKY)

In [ ]:
# Mount Google Drive (guarded by DRIVE_MOUNT). Skip it if you uploaded the bundle
# directly and keep OUTPUT_DIR on local disk.
if DRIVE_MOUNT:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("DRIVE_MOUNT is False — using the uploaded BUNDLE_PATH and local OUTPUT_DIR only")

In [ ]:
# Unpack the prepared bundle and validate the dataset before training.
import os
import subprocess
import tarfile

os.makedirs(DATA_ROOT, exist_ok=True)
with tarfile.open(BUNDLE_PATH) as tar:
    # export-colab-bundle nests every member under a single top-level
    # `allsky_bundle/` directory, so extracting into DATA_ROOT lands the
    # manifest.parquet + .meta.json + splits.json + embeddings/ + config/
    # under BUNDLE_ROOT.
    tar.extractall(DATA_ROOT, filter="data")

subprocess.run(
    [ALLSKY, "validate-dataset", "--manifest", f"{BUNDLE_ROOT}/manifest.parquet"],
    cwd=WORKDIR,
    check=True,
)

In [ ]:
# Live metrics (TensorBoard reads OUTPUT_DIR/runs recursively).
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR

In [ ]:
# Train (resumable): --resume auto picks up OUTPUT_DIR/last.ckpt after a Colab
# disconnect. Re-run this cell to continue an interrupted run.
import subprocess

subprocess.run(
    [
        ALLSKY,
        "train",
        "--config",
        EXPERIMENT_CONFIG,
        "--data-root",
        BUNDLE_ROOT,
        "--out-dir",
        OUTPUT_DIR,
        "--device",
        "cuda",
        "--amp",
        "--resume",
        "auto",
    ],
    cwd=WORKDIR,
    check=True,
)

In [ ]:
# Evaluate the best checkpoint on the held-out TEST split (writes a report dir
# under OUTPUT_DIR/eval-test: metrics.json, stratified.csv, report.md, predictions).
import subprocess

subprocess.run(
    [
        ALLSKY,
        "evaluate",
        "--checkpoint",
        f"{OUTPUT_DIR}/best.ckpt",
        "--split",
        "test",
        "--data-root",
        BUNDLE_ROOT,
    ],
    cwd=WORKDIR,
    check=True,
)

In [ ]:
# Persist the run to Drive (checkpoints + metrics + TB + eval report). Run this
# periodically during long trainings and once at the end.
import os
import shutil

if DRIVE_MOUNT:
    dest = os.path.join(DRIVE_DIR, os.path.basename(OUTPUT_DIR))
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
    print("copied", OUTPUT_DIR, "->", dest)
else:
    print("DRIVE_MOUNT is False — nothing copied; OUTPUT_DIR stays on local disk")

## Troubleshooting

- **Resume after a disconnect.** Re-run the clone/bootstrap/mount cells, then the
  train cell — `--resume auto` continues from `OUTPUT_DIR/last.ckpt` and never
  overwrites a better `best.ckpt`. If `OUTPUT_DIR` is local (not Drive), copy the
  last backup from `DRIVE_DIR` into `OUTPUT_DIR` first.
- **No GPU / `--device cuda` fails.** Runtime → Change runtime type → GPU. The
  `[allsky]` extra installs a **CPU** torch wheel; for GPU you must install a CUDA
  torch build into the venv (see the commented line in the bootstrap cell), or
  swap `--device cuda --amp` for `--device cpu --no-amp` (slow — smoke tests only).
- **Bundle path.** `BUNDLE_PATH` must point at an `allsky export-colab-bundle`
  archive. Build it locally with
  `allsky export-colab-bundle -o bundle.tar.gz --config configs/allsky/data/local_prepare.yaml`
  and drop it on Drive (or use the Colab file uploader and set `DRIVE_MOUNT = False`).
- **Image-mode (`v6_film_finetune`).** This decodes JPEGs and finetunes DINOv2, so
  the bundle must include the frames and a real DINOv2 backbone downloads on first
  use — heavier and GPU-only. The embedding experiments (V0–V5, V7) need only the
  precomputed embeddings shipped in the bundle.
